# Notebook 11 — Exact-state DFT preparation and hypothesis-driven case manifest

**Private execution notebook.** This notebook prepares exact charged/discharged endpoint structures for first-principles spot checks.

## Scientific purpose

This notebook:

1. verifies the successful Notebook 06 decision;
2. traces each DFT case to the exact Notebook 01 insertion-electrode record;
3. uses exact `id_charge` and `id_discharge` values, including modern alphanumeric Materials Project IDs;
4. extracts structures from the authoritative raw Notebook 01 insertion-electrode JSON;
5. rejects formula-first structure substitution;
6. writes exact structure files and VASP input templates without licensed POTCAR files;
7. creates a hypothesis-driven case manifest for pilot DFT execution.

It performs **no DFT calculation**, **no new model training**, **no candidate ranking**, and **no Materials Project API query**.

## Required inputs

Place beside this notebook:

```text
canonical Notebook 01 repository artifacts
canonical Notebook 06 repository artifacts
```

Optional enrichment inputs:

```text
canonical Notebook 05 repository artifacts
canonical Notebook 07 repository artifacts
canonical Notebook 08 repository artifacts
canonical Notebook 09 repository artifacts
```

Already extracted `outputs/notebook_XX/` directories are also supported.

## Required final decision

```text
FULL_GO_TO_PILOT_DFT
```

A restricted decision is acceptable only when at least three primary candidate pairs are exact, ordered, manageable, and fully prepared.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import shutil
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from pymatgen.core import Composition, Structure
    from pymatgen.io.cif import CifWriter
    from pymatgen.io.vasp import Poscar
    from pymatgen.io.vasp.sets import MPRelaxSet, MPStaticSet
    import pymatgen
except Exception as exc:
    raise ImportError(
        "Notebook 11 requires pymatgen. Install it in the active environment with: "
        "%pip install pymatgen monty"
    ) from exc

def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

ROOT = REPOSITORY_ROOT
OUTPUT_ROOT = artifact_namespace("11", REPOSITORY_ROOT)
AUDIT_DIR = OUTPUT_ROOT / "audit"
PROCESSED_DIR = OUTPUT_ROOT / "processed"
METADATA_DIR = OUTPUT_ROOT / "metadata"
STRUCTURE_DIR = OUTPUT_ROOT / "exact_structures"
DFT_INPUT_DIR = OUTPUT_ROOT / "dft_inputs"
LOG_DIR = OUTPUT_ROOT / "logs"
CACHE_DIR = runtime_cache_root(REPOSITORY_ROOT) / "notebook_11_exact_state_inputs"

for directory in [
    AUDIT_DIR, PROCESSED_DIR, METADATA_DIR,
    STRUCTURE_DIR, DFT_INPUT_DIR, LOG_DIR, CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PRIMARY_CANDIDATES = [
    "Na_candidate_06",
    "Na_candidate_10",
    "Na_candidate_11",
    "Na_candidate_15",
    "Na_candidate_16",
]
OPTIONAL_CONTROL = ["Na_candidate_02"]

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("pymatgen:", getattr(pymatgen, "__version__", "unknown"))
print("Output:", OUTPUT_ROOT)

In [ ]:
# ============================================================
# Canonical clean-room input resolution
# ============================================================

NB01 = artifact_namespace("01", REPOSITORY_ROOT)
NB05 = artifact_namespace("05", REPOSITORY_ROOT)
NB06 = artifact_namespace("06", REPOSITORY_ROOT)
NB08 = artifact_namespace("08", REPOSITORY_ROOT)
NB09 = artifact_namespace("09", REPOSITORY_ROOT)


def resolve_input(
    namespace,
    relative_suffix: str,
    required: bool = True,
) -> Path | None:
    path = namespace / relative_suffix
    if path.exists():
        return path
    if required:
        raise FileNotFoundError(f"Required canonical input not found: {path}")
    return None


INPUTS = {
    "nb08_core": resolve_input(
        NB01,
        "processed/01_multion_insertion_electrodes_core_with_groups.csv",
    ),
    "nb08_raw_na": resolve_input(
        NB01,
        "raw/mp_Na_insertion_electrodes_raw.json",
        required=False,
    ),
    "nb08_decision": resolve_input(
        NB01,
        "metadata/01_final_decision.json",
    ),
    "nb10c_decision": resolve_input(
        NB06,
        "metadata/06_final_decision.json",
    ),
}

RUNTIME_GENERATED_INPUT_POLICY = {
    "nb08_raw_na": {
        "status": "runtime_generated_not_bundled",
        "environment_variable": "CMT_RAW_NA_JSON",
        "default_repository_path": str(
            NB01 / "raw/mp_Na_insertion_electrodes_raw.json"
        ),
        "substitution_policy": "do_not_replace_with_processed_csv",
    }
}

OPTIONAL_INPUTS = {
    "nb10b_predictions": resolve_input(
        NB05,
        "processed/05_physics_constrained_oof_predictions.csv",
        required=False,
    ),
    "nb11_predictions": (
        lambda path: path if path.exists() else None
    )(
        Path(
            os.environ.get(
                "CMT_EXTERNAL_DATA_DIR",
                str(REPOSITORY_ROOT / "external_data"),
            )
        )
        / "07_uncertainty_predictions.csv"
    ),
    "nb12_candidates": resolve_input(
        NB08,
        "processed/08_sodium_top30_candidates_for_literature_review.csv",
        required=False,
    ),
    "nb13_summary": resolve_input(
        NB09,
        "processed/09_candidate_literature_analogue_summary.csv",
        required=False,
    ),
}

input_rows = []
for role, path in {**INPUTS, **OPTIONAL_INPUTS}.items():
    input_rows.append({
        "role": role,
        "available": path is not None and path.exists(),
        "path": str(path.resolve()) if path is not None and path.exists() else "",
        "sha256": sha256(path) if path is not None and path.exists() else "",
    })

input_manifest = pd.DataFrame(input_rows)
input_manifest.to_csv(METADATA_DIR / "11_input_file_hashes.csv", index=False)
display(input_manifest)


In [ ]:
# ============================================================
# Preflight decisions and locked exact mapping
# ============================================================

decision08 = json.loads(INPUTS["nb08_decision"].read_text(encoding="utf-8"))
decision10c = json.loads(INPUTS["nb10c_decision"].read_text(encoding="utf-8"))

accepted08 = decision08.get("final_decision") in {
    "FULL_GO_TO_NOTEBOOK_09",
    "CONDITIONAL_GO_MODIFY_PATH_A",
}
accepted10c = (
    decision10c.get("final_decision")
    == "FULL_GO_TO_EXACT_STATE_DFT_PREPARATION"
)

LOCKED_MAPPING = pd.DataFrame([
    {
        "candidate_id": "Na_candidate_06",
        "selection_role": "primary",
        "record_index_expected": "Na_000056",
        "electrode_uid_expected": "el_7866a4da483e62fa",
        "id_charge_expected": "mp-aaackrry",
        "id_discharge_expected": "mp-aaabrwlw",
        "dft_test_role": "carbonophosphate_reference_case",
    },
    {
        "candidate_id": "Na_candidate_10",
        "selection_role": "primary",
        "record_index_expected": "Na_000073",
        "electrode_uid_expected": "el_31d54be5f7dfc2b4",
        "id_charge_expected": "mp-aaabrunh",
        "id_discharge_expected": "mp-aaaabgtj",
        "dft_test_role": "same_framework_transition_metal_substitution_Fe",
    },
    {
        "candidate_id": "Na_candidate_11",
        "selection_role": "primary",
        "record_index_expected": "Na_000087",
        "electrode_uid_expected": "el_21648b3822f38a5a",
        "id_charge_expected": "mp-aaabrhtw",
        "id_discharge_expected": "mp-aaaabcmv",
        "dft_test_role": "same_framework_transition_metal_substitution_Mn",
    },
    {
        "candidate_id": "Na_candidate_15",
        "selection_role": "primary",
        "record_index_expected": "Na_000120",
        "electrode_uid_expected": "el_eeae341957dac1d3",
        "id_charge_expected": "mp-aaaabnjm",
        "id_discharge_expected": "mp-aaacrmqw",
        "dft_test_role": "cross_family_NASICON_like_case",
    },
    {
        "candidate_id": "Na_candidate_16",
        "selection_role": "primary",
        "record_index_expected": "Na_000189",
        "electrode_uid_expected": "el_3992a779a57c4f8f",
        "id_charge_expected": "mp-aaabhhug",
        "id_discharge_expected": "mp-aaacqeli",
        "dft_test_role": "cross_family_pyrophosphate_case",
    },
    {
        "candidate_id": "Na_candidate_02",
        "selection_role": "optional_control",
        "record_index_expected": "Na_000286",
        "electrode_uid_expected": "el_218b2b75403fd262",
        "id_charge_expected": "mp-aaacupmn",
        "id_discharge_expected": "mp-aaabxjhz",
        "dft_test_role": "transition_metal_oxide_control",
    },
])

LOCKED_MAPPING.to_csv(
    PROCESSED_DIR / "11_locked_exact_endpoint_mapping.csv",
    index=False,
)

preflight = pd.DataFrame([
    {"check": "notebook08_decision_accepted", "pass": bool(accepted08)},
    {"check": "notebook10C_decision_accepted", "pass": bool(accepted10c)},
    {
        "check": "modern_alphanumeric_mp_ids_preserved",
        "pass": bool(
            LOCKED_MAPPING["id_charge_expected"]
            .str.fullmatch(r"mp-[A-Za-z0-9]+")
            .all()
            and LOCKED_MAPPING["id_discharge_expected"]
            .str.fullmatch(r"mp-[A-Za-z0-9]+")
            .all()
        ),
    },
])
preflight.to_csv(AUDIT_DIR / "11_preflight_audit.csv", index=False)
display(preflight)

if not preflight["pass"].all():
    raise RuntimeError("Notebook 11 preflight failed")

In [ ]:
# ============================================================
# Validate mapping against the authoritative Notebook 01 core table
# ============================================================

core = pd.read_csv(INPUTS["nb08_core"], low_memory=False)

required_columns = {
    "record_index", "electrode_uid", "working_ion",
    "id_charge", "id_discharge",
    "formula_charge", "formula_discharge",
    "battery_formula", "framework_formula",
}
missing_columns = sorted(required_columns - set(core.columns))
if missing_columns:
    raise KeyError(f"Notebook 01 core table is missing: {missing_columns}")

audit_rows = []
resolved_core_rows = []

for mapping in LOCKED_MAPPING.itertuples(index=False):
    matches = core[
        (core["record_index"].astype(str) == str(mapping.record_index_expected))
        & (core["electrode_uid"].astype(str) == str(mapping.electrode_uid_expected))
    ].copy()

    row = matches.iloc[0] if len(matches) == 1 else None
    pass_mapping = bool(
        row is not None
        and str(row["working_ion"]) == "Na"
        and str(row["id_charge"]) == mapping.id_charge_expected
        and str(row["id_discharge"]) == mapping.id_discharge_expected
    )

    audit_rows.append({
        "candidate_id": mapping.candidate_id,
        "n_core_matches": int(len(matches)),
        "record_index_match": bool(
            row is not None
            and str(row["record_index"]) == mapping.record_index_expected
        ),
        "electrode_uid_match": bool(
            row is not None
            and str(row["electrode_uid"]) == mapping.electrode_uid_expected
        ),
        "id_charge_match": bool(
            row is not None
            and str(row["id_charge"]) == mapping.id_charge_expected
        ),
        "id_discharge_match": bool(
            row is not None
            and str(row["id_discharge"]) == mapping.id_discharge_expected
        ),
        "pass": pass_mapping,
    })

    if row is not None:
        record = row.to_dict()
        record.update({
            "candidate_id": mapping.candidate_id,
            "selection_role": mapping.selection_role,
            "dft_test_role": mapping.dft_test_role,
        })
        resolved_core_rows.append(record)

core_mapping_audit = pd.DataFrame(audit_rows)
core_mapping_audit.to_csv(
    AUDIT_DIR / "11_core_mapping_audit.csv",
    index=False,
)
display(core_mapping_audit)

if not core_mapping_audit["pass"].all():
    raise RuntimeError(
        "Exact Notebook 01 core mapping failed. "
        "Do not use formula-based fallback."
    )

resolved_core = pd.DataFrame(resolved_core_rows)
resolved_core.to_csv(
    PROCESSED_DIR / "11_exact_candidate_core_records.csv",
    index=False,
)

In [ ]:
# ============================================================
# Load raw Na insertion-electrode JSON and locate exact raw records
# ============================================================

def load_json_flexible(path: Path) -> Any:
    text = path.read_text(encoding="utf-8")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        records = []
        for line in text.splitlines():
            line = line.strip()
            if line:
                records.append(json.loads(line))
        return records


def normalize_record_list(raw: Any) -> list[dict]:
    if isinstance(raw, list):
        return [item for item in raw if isinstance(item, dict)]

    if isinstance(raw, dict):
        for key in ["data", "records", "results", "items", "docs"]:
            value = raw.get(key)
            if isinstance(value, list):
                return [item for item in value if isinstance(item, dict)]

        # A dictionary indexed by record labels.
        dict_values = list(raw.values())
        if dict_values and all(isinstance(value, dict) for value in dict_values):
            return dict_values

    raise TypeError("Could not identify the raw insertion-electrode record list")


def contains_exact_token(obj: Any, token: str) -> bool:
    if isinstance(obj, str):
        return obj == token
    if isinstance(obj, dict):
        return any(contains_exact_token(value, token) for value in obj.values())
    if isinstance(obj, list):
        return any(contains_exact_token(value, token) for value in obj)
    return False


raw_na = load_json_flexible(INPUTS["nb08_raw_na"])
raw_records = normalize_record_list(raw_na)

resolution_rows = []
raw_match_by_candidate = {}

for mapping in LOCKED_MAPPING.itertuples(index=False):
    matches = [
        (index, record)
        for index, record in enumerate(raw_records)
        if contains_exact_token(record, mapping.id_charge_expected)
        and contains_exact_token(record, mapping.id_discharge_expected)
    ]

    raw_match_by_candidate[mapping.candidate_id] = matches[0][1] if len(matches) == 1 else None

    resolution_rows.append({
        "candidate_id": mapping.candidate_id,
        "id_charge": mapping.id_charge_expected,
        "id_discharge": mapping.id_discharge_expected,
        "n_raw_records_containing_both_exact_ids": int(len(matches)),
        "raw_record_position": int(matches[0][0]) if len(matches) == 1 else np.nan,
        "resolution_method": "both_exact_endpoint_ids_in_same_raw_record",
        "pass": len(matches) == 1,
    })

raw_record_audit = pd.DataFrame(resolution_rows)
raw_record_audit.to_csv(
    AUDIT_DIR / "11_raw_record_resolution_audit.csv",
    index=False,
)
display(raw_record_audit)

if not raw_record_audit["pass"].all():
    raise RuntimeError(
        "Exact raw-record resolution failed. "
        "Formula search is intentionally disabled."
    )

In [ ]:
# ============================================================
# Extract exact endpoint structures from raw records
# ============================================================

ID_KEYS = {
    "entry_id", "material_id", "task_id", "mp_id",
    "id", "identifier", "entry_id_deprecated",
}
STRUCTURE_KEYS = {
    "structure", "final_structure", "initial_structure",
    "host_structure", "charged_structure", "discharged_structure",
}


def looks_like_structure_dict(obj: Any) -> bool:
    return (
        isinstance(obj, dict)
        and "lattice" in obj
        and "sites" in obj
        and isinstance(obj["sites"], list)
    )


def scalar_equals(value: Any, token: str) -> bool:
    if isinstance(value, str):
        return value == token
    if isinstance(value, (list, tuple)):
        return any(scalar_equals(item, token) for item in value)
    return False


def walk_dicts(obj: Any, path: tuple = ()):
    if isinstance(obj, dict):
        yield path, obj
        for key, value in obj.items():
            yield from walk_dicts(value, path + (str(key),))
    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            yield from walk_dicts(value, path + (str(index),))


def collect_structure_candidates(record: dict, target_id: str) -> list[dict]:
    candidates = []

    for path, node in walk_dicts(record):
        direct_id_keys = [
            key for key, value in node.items()
            if str(key).lower() in ID_KEYS and scalar_equals(value, target_id)
        ]

        if not direct_id_keys:
            continue

        for key, value in node.items():
            if str(key).lower() in STRUCTURE_KEYS and looks_like_structure_dict(value):
                candidates.append({
                    "path": "/".join(path + (str(key),)),
                    "id_key": direct_id_keys[0],
                    "structure_dict": value,
                    "association": "direct_id_and_structure_siblings",
                })

        if looks_like_structure_dict(node):
            candidates.append({
                "path": "/".join(path),
                "id_key": direct_id_keys[0],
                "structure_dict": node,
                "association": "id_inside_structure_node",
            })

    # Conservative fallback: an exact-ID-containing parent with one immediate structure child.
    if not candidates:
        for path, node in walk_dicts(record):
            if not contains_exact_token(node, target_id):
                continue
            immediate = [
                (key, value)
                for key, value in node.items()
                if looks_like_structure_dict(value)
            ]
            if len(immediate) == 1:
                key, value = immediate[0]
                candidates.append({
                    "path": "/".join(path + (str(key),)),
                    "id_key": "exact_id_in_parent_subtree",
                    "structure_dict": value,
                    "association": "conservative_parent_subtree",
                })

    # Deduplicate by canonical JSON.
    unique = {}
    for candidate in candidates:
        signature = json.dumps(
            candidate["structure_dict"],
            sort_keys=True,
            separators=(",", ":"),
        )
        unique.setdefault(signature, candidate)
    return list(unique.values())


def normalized_composition(formula_or_composition):
    composition = (
        formula_or_composition
        if isinstance(formula_or_composition, Composition)
        else Composition(str(formula_or_composition))
    )
    return composition.fractional_composition


def formula_matches_structure(expected_formula: str, structure: Structure) -> bool:
    try:
        expected = normalized_composition(expected_formula)
        observed = normalized_composition(structure.composition)
        return bool(expected.almost_equals(observed, rtol=1e-6, atol=1e-8))
    except Exception:
        return False


structure_rows = []
selected_structures = {}

for mapping in LOCKED_MAPPING.itertuples(index=False):
    core_row = resolved_core.loc[
        resolved_core["candidate_id"] == mapping.candidate_id
    ].iloc[0]
    raw_record = raw_match_by_candidate[mapping.candidate_id]

    for state_role, endpoint_id, formula_column in [
        ("charged", mapping.id_charge_expected, "formula_charge"),
        ("discharged", mapping.id_discharge_expected, "formula_discharge"),
    ]:
        expected_formula = str(core_row[formula_column])
        candidates = collect_structure_candidates(raw_record, endpoint_id)

        parsed = []
        for candidate in candidates:
            try:
                structure = Structure.from_dict(candidate["structure_dict"])
                parsed.append({
                    **candidate,
                    "structure": structure,
                    "formula_match": formula_matches_structure(
                        expected_formula,
                        structure,
                    ),
                })
            except Exception as exc:
                structure_rows.append({
                    "candidate_id": mapping.candidate_id,
                    "selection_role": mapping.selection_role,
                    "state_role": state_role,
                    "endpoint_id": endpoint_id,
                    "expected_formula": expected_formula,
                    "candidate_path": candidate["path"],
                    "parse_success": False,
                    "formula_match": False,
                    "selected": False,
                    "parse_error": repr(exc),
                })

        formula_matches = [item for item in parsed if item["formula_match"]]
        if len(formula_matches) == 1:
            selected = formula_matches[0]
            selection_reason = "unique_exact_id_plus_formula_match"
        elif len(parsed) == 1:
            selected = parsed[0]
            selection_reason = "unique_exact_id_structure_candidate"
        else:
            selected = None
            selection_reason = "ambiguous_or_missing_structure"

        if selected is not None:
            structure = selected["structure"]
            selected_structures[(mapping.candidate_id, state_role)] = structure

        for item in parsed:
            structure = item["structure"]
            is_selected = bool(item is selected)
            structure_rows.append({
                "candidate_id": mapping.candidate_id,
                "selection_role": mapping.selection_role,
                "state_role": state_role,
                "endpoint_id": endpoint_id,
                "expected_formula": expected_formula,
                "candidate_path": item["path"],
                "association": item["association"],
                "parse_success": True,
                "formula_match": bool(item["formula_match"]),
                "selected": is_selected,
                "selection_reason": selection_reason if is_selected else "",
                "structure_formula": structure.composition.reduced_formula,
                "nsites": int(len(structure)),
                "volume": float(structure.volume),
                "is_ordered": bool(structure.is_ordered),
                "parse_error": "",
            })

structure_audit = pd.DataFrame(structure_rows)
structure_audit.to_csv(
    AUDIT_DIR / "11_structure_parse_and_identity_audit.csv",
    index=False,
)

selected_audit = structure_audit[structure_audit["selected"] == True].copy()
display(selected_audit)

expected_state_count = 2 * len(LOCKED_MAPPING)
if len(selected_audit) != expected_state_count:
    print(
        f"Warning: resolved {len(selected_audit)} of "
        f"{expected_state_count} expected candidate states."
    )

In [ ]:
# ============================================================
# Write exact structure files and provenance
# ============================================================

structure_manifest_rows = []

for mapping in LOCKED_MAPPING.itertuples(index=False):
    for state_role, endpoint_id in [
        ("charged", mapping.id_charge_expected),
        ("discharged", mapping.id_discharge_expected),
    ]:
        structure = selected_structures.get((mapping.candidate_id, state_role))
        state_dir = STRUCTURE_DIR / mapping.candidate_id / state_role
        state_dir.mkdir(parents=True, exist_ok=True)

        if structure is None:
            structure_manifest_rows.append({
                "candidate_id": mapping.candidate_id,
                "selection_role": mapping.selection_role,
                "state_role": state_role,
                "endpoint_id": endpoint_id,
                "written": False,
                "reason": "exact_structure_not_resolved",
            })
            continue

        structure_json = state_dir / "structure.json"
        poscar_path = state_dir / "POSCAR"
        cif_path = state_dir / "structure.cif"
        provenance_path = state_dir / "provenance.json"

        structure_json.write_text(
            json.dumps(structure.as_dict(), indent=2),
            encoding="utf-8",
        )
        Poscar(structure).write_file(poscar_path)
        CifWriter(structure).write_file(cif_path)

        provenance = {
            "candidate_id": mapping.candidate_id,
            "selection_role": mapping.selection_role,
            "record_index": mapping.record_index_expected,
            "electrode_uid": mapping.electrode_uid_expected,
            "state_role": state_role,
            "exact_endpoint_id": endpoint_id,
            "resolution_method": "exact_endpoint_id_in_authoritative_notebook08_raw_record",
            "source_raw_json_sha256": sha256(INPUTS["nb08_raw_na"]),
            "structure_formula": structure.composition.reduced_formula,
            "nsites": len(structure),
            "volume": structure.volume,
            "is_ordered": structure.is_ordered,
            "generated_utc": RUN_TIMESTAMP_UTC,
        }
        provenance_path.write_text(
            json.dumps(provenance, indent=2),
            encoding="utf-8",
        )

        structure_manifest_rows.append({
            "candidate_id": mapping.candidate_id,
            "selection_role": mapping.selection_role,
            "state_role": state_role,
            "endpoint_id": endpoint_id,
            "written": True,
            "structure_formula": structure.composition.reduced_formula,
            "nsites": int(len(structure)),
            "volume": float(structure.volume),
            "is_ordered": bool(structure.is_ordered),
            "structure_json": str(structure_json.relative_to(OUTPUT_ROOT)),
            "poscar": str(poscar_path.relative_to(OUTPUT_ROOT)),
            "cif": str(cif_path.relative_to(OUTPUT_ROOT)),
            "provenance": str(provenance_path.relative_to(OUTPUT_ROOT)),
            "structure_sha256": sha256(structure_json),
        })

structure_manifest = pd.DataFrame(structure_manifest_rows)
structure_manifest.to_csv(
    PROCESSED_DIR / "11_exact_structure_manifest.csv",
    index=False,
)
display(structure_manifest)

In [ ]:
# ============================================================
# Generate VASP-compatible relaxation/static templates without POTCAR files
# ============================================================

RELAX_SETTINGS = {
    "ENCUT": 520,
    "EDIFF": 1e-5,
    "EDIFFG": -0.03,
    "ISIF": 3,
    "NSW": 120,
    "ISPIN": 2,
    "LASPH": True,
    "LREAL": False,
}

STATIC_SETTINGS = {
    "ENCUT": 520,
    "EDIFF": 1e-6,
    "NSW": 0,
    "IBRION": -1,
    "ISMEAR": -5,
    "ISPIN": 2,
    "LASPH": True,
    "LREAL": False,
}

vasp_rows = []

for mapping in LOCKED_MAPPING.itertuples(index=False):
    for state_role, endpoint_id in [
        ("charged", mapping.id_charge_expected),
        ("discharged", mapping.id_discharge_expected),
    ]:
        structure = selected_structures.get((mapping.candidate_id, state_role))
        if structure is None:
            vasp_rows.append({
                "candidate_id": mapping.candidate_id,
                "selection_role": mapping.selection_role,
                "state_role": state_role,
                "endpoint_id": endpoint_id,
                "template_written": False,
                "error": "exact_structure_not_resolved",
            })
            continue

        base = DFT_INPUT_DIR / mapping.candidate_id / state_role
        relax_dir = base / "01_relax"
        static_dir = base / "02_static_template"
        relax_dir.mkdir(parents=True, exist_ok=True)
        static_dir.mkdir(parents=True, exist_ok=True)

        try:
            relax_set = MPRelaxSet(
                structure,
                force_gamma=True,
                user_incar_settings=RELAX_SETTINGS,
            )
            relax_set.write_input(
                relax_dir,
                potcar_spec=True,
            )

            static_set = MPStaticSet(
                structure,
                force_gamma=True,
                user_incar_settings=STATIC_SETTINGS,
            )
            static_set.write_input(
                static_dir,
                potcar_spec=True,
            )

            # Avoid accidental use of the initial structure as a production static input.
            static_poscar = static_dir / "POSCAR"
            if static_poscar.exists():
                static_poscar.rename(static_dir / "POSCAR_INITIAL_REFERENCE")

            static_readme = static_dir / "README_STATIC_STAGE.txt"
            static_readme.write_text(
                "This is a static-calculation template.\n"
                "After a successful relaxation, copy the converged 01_relax/CONTCAR "
                "to this directory as POSCAR.\n"
                "Do not use POSCAR_INITIAL_REFERENCE as the final production static structure.\n",
                encoding="utf-8",
            )

            for directory in [relax_dir, static_dir]:
                forbidden_potcar = directory / "POTCAR"
                if forbidden_potcar.exists():
                    forbidden_potcar.unlink()

            vasp_rows.append({
                "candidate_id": mapping.candidate_id,
                "selection_role": mapping.selection_role,
                "state_role": state_role,
                "endpoint_id": endpoint_id,
                "template_written": True,
                "relax_dir": str(relax_dir.relative_to(OUTPUT_ROOT)),
                "static_template_dir": str(static_dir.relative_to(OUTPUT_ROOT)),
                "potcar_spec_present": bool(
                    (relax_dir / "POTCAR.spec").exists()
                    and (static_dir / "POTCAR.spec").exists()
                ),
                "licensed_potcar_present": bool(
                    (relax_dir / "POTCAR").exists()
                    or (static_dir / "POTCAR").exists()
                ),
                "error": "",
            })
        except Exception as exc:
            vasp_rows.append({
                "candidate_id": mapping.candidate_id,
                "selection_role": mapping.selection_role,
                "state_role": state_role,
                "endpoint_id": endpoint_id,
                "template_written": False,
                "error": repr(exc),
            })

vasp_manifest = pd.DataFrame(vasp_rows)
vasp_manifest.to_csv(
    PROCESSED_DIR / "11_vasp_input_template_manifest.csv",
    index=False,
)
display(vasp_manifest)

In [ ]:
# ============================================================
# Optional evidence enrichment and hypothesis-driven case manifest
# ============================================================

case_manifest = resolved_core[
    [
        "candidate_id", "selection_role", "dft_test_role",
        "record_index", "electrode_uid",
        "battery_formula", "framework_formula",
        "formula_charge", "formula_discharge",
        "average_voltage", "capacity_grav", "energy_grav",
        "max_delta_volume", "stability_worst",
        "id_charge", "id_discharge",
    ]
].copy()

case_manifest["exact_pair_resolved"] = case_manifest["candidate_id"].map(
    structure_manifest.groupby("candidate_id")["written"].all()
)
case_manifest["ordered_pair"] = case_manifest["candidate_id"].map(
    structure_manifest.groupby("candidate_id")["is_ordered"].all()
)
case_manifest["max_nsites"] = case_manifest["candidate_id"].map(
    structure_manifest.groupby("candidate_id")["nsites"].max()
)
case_manifest["manageable_cell"] = (
    pd.to_numeric(case_manifest["max_nsites"], errors="coerce") <= 200
)
case_manifest["vasp_pair_written"] = case_manifest["candidate_id"].map(
    vasp_manifest.groupby("candidate_id")["template_written"].all()
)

# Add candidate-level annotations when optional files are available.
if OPTIONAL_INPUTS["nb12_candidates"] is not None:
    nb12 = pd.read_csv(OPTIONAL_INPUTS["nb12_candidates"], low_memory=False)
    candidate_column = next(
        (
            column for column in [
                "candidate_id", "manual_candidate_id", "candidate_label"
            ]
            if column in nb12.columns
        ),
        None,
    )
    if candidate_column:
        selected_columns = [candidate_column]
        for column in [
            "rank_robustness", "top10_probability", "top20_probability",
            "domain_class", "uncertainty_penalty", "pareto_any",
        ]:
            if column in nb12.columns:
                selected_columns.append(column)
        enrichment = (
            nb12[selected_columns]
            .drop_duplicates(candidate_column)
            .rename(columns={candidate_column: "candidate_id"})
        )
        case_manifest = case_manifest.merge(
            enrichment,
            on="candidate_id",
            how="left",
        )

if OPTIONAL_INPUTS["nb13_summary"] is not None:
    nb13 = pd.read_csv(OPTIONAL_INPUTS["nb13_summary"], low_memory=False)
    candidate_column = next(
        (
            column for column in [
                "candidate_id", "manual_candidate_id", "candidate_label"
            ]
            if column in nb13.columns
        ),
        None,
    )
    if candidate_column:
        selected_columns = [candidate_column]
        for column in [
            "best_analogue_category", "n_checked_rows",
            "n_exact_or_near_exact", "needs_second_check",
        ]:
            if column in nb13.columns:
                selected_columns.append(column)
        enrichment = (
            nb13[selected_columns]
            .drop_duplicates(candidate_column)
            .rename(columns={candidate_column: "candidate_id"})
        )
        case_manifest = case_manifest.merge(
            enrichment,
            on="candidate_id",
            how="left",
        )

case_manifest["pilot_priority"] = np.where(
    case_manifest["selection_role"].eq("primary"),
    "primary",
    "optional_control",
)
case_manifest["scientific_hypothesis"] = case_manifest["dft_test_role"].map({
    "carbonophosphate_reference_case":
        "Establish a reference exact-state voltage/volume result for the carbonophosphate series.",
    "same_framework_transition_metal_substitution_Fe":
        "Test whether changing the transition metal within the same framework alters DFT/ML agreement.",
    "same_framework_transition_metal_substitution_Mn":
        "Test framework-controlled chemical transfer for a second transition-metal substitution.",
    "cross_family_NASICON_like_case":
        "Test cross-family transfer for a structurally distinct phosphate framework.",
    "cross_family_pyrophosphate_case":
        "Test cross-family transfer and volume response in a pyrophosphate framework.",
    "transition_metal_oxide_control":
        "Provide an optional oxide control outside the primary polyanion set.",
})

case_manifest.to_csv(
    PROCESSED_DIR / "11_hypothesis_driven_dft_case_manifest.csv",
    index=False,
)
display(case_manifest)

In [ ]:
# ============================================================
# DFT protocol requirements and pilot execution order
# ============================================================

protocol_rows = [
    {
        "item": "DFT code",
        "required_value": "VASP or rigorously documented equivalent",
        "status": "REQUIRES_USER_RESOURCE_CONFIRMATION",
        "notes": "Templates are VASP-compatible; no calculation has been run.",
    },
    {
        "item": "exchange_correlation",
        "required_value": "PBE with Materials-Project-compatible corrections where applicable",
        "status": "VERIFY_BEFORE_PRODUCTION",
        "notes": "Use the same functional and correction policy for all endpoint and Na-reference calculations.",
    },
    {
        "item": "Hubbard_U",
        "required_value": "Inspect generated INCAR/POTCAR.spec and Notebook 01 raw metadata",
        "status": "VERIFY_PER_COMPOUND",
        "notes": "Do not mix incompatible U settings across charged/discharged endpoints.",
    },
    {
        "item": "spin",
        "required_value": "ISPIN=2 with chemistry-appropriate initial magnetic moments",
        "status": "VERIFY_PER_COMPOUND",
        "notes": "Test alternative magnetic initializations when convergence is unstable.",
    },
    {
        "item": "plane_wave_cutoff",
        "required_value": "ENCUT=520 eV minimum template value",
        "status": "PILOT_CONVERGENCE_REQUIRED",
        "notes": "Confirm against the selected PAW datasets.",
    },
    {
        "item": "k_point_sampling",
        "required_value": "MPRelaxSet/MPStaticSet generated mesh",
        "status": "PILOT_CONVERGENCE_REQUIRED",
        "notes": "Confirm total-energy and voltage convergence.",
    },
    {
        "item": "relaxation",
        "required_value": "EDIFF=1e-5, EDIFFG=-0.03, ISIF=3, NSW=120",
        "status": "TEMPLATE_ONLY",
        "notes": "Extend NSW or tighten thresholds when necessary.",
    },
    {
        "item": "static_stage",
        "required_value": "Use relaxed CONTCAR as static POSCAR",
        "status": "MANDATORY",
        "notes": "The notebook renames the initial static POSCAR to prevent accidental use.",
    },
    {
        "item": "sodium_reference",
        "required_value": "metallic Na calculated with exactly compatible settings",
        "status": "MANDATORY",
        "notes": "Required for average-voltage calculation.",
    },
    {
        "item": "licensed_files",
        "required_value": "Never commit POTCAR files",
        "status": "MANDATORY",
        "notes": "Only POTCAR.spec files may enter the private execution package or public release.",
    },
]
protocol_table = pd.DataFrame(protocol_rows)
protocol_table.to_csv(
    PROCESSED_DIR / "11_dft_protocol_requirements.csv",
    index=False,
)

pilot_order = pd.DataFrame([
    {
        "pilot_order": 1,
        "candidate_id": "Na_candidate_06",
        "purpose": "first complete charged/discharged pair and Na-reference workflow validation",
    },
    {
        "pilot_order": 2,
        "candidate_id": "Na_candidate_15",
        "purpose": "cross-family protocol validation after the first pair succeeds",
    },
    {
        "pilot_order": 3,
        "candidate_id": "Na_candidate_10",
        "purpose": "same-framework transition-metal substitution comparison",
    },
    {
        "pilot_order": 4,
        "candidate_id": "Na_candidate_11",
        "purpose": "second same-framework substitution comparison",
    },
    {
        "pilot_order": 5,
        "candidate_id": "Na_candidate_16",
        "purpose": "pyrophosphate cross-family comparison",
    },
    {
        "pilot_order": 6,
        "candidate_id": "Na_candidate_02",
        "purpose": "optional oxide control only after the primary cases are stable",
    },
])
pilot_order.to_csv(
    PROCESSED_DIR / "11_pilot_dft_execution_order.csv",
    index=False,
)

display(protocol_table)
display(pilot_order)

In [ ]:
# ============================================================
# Final audits, secret/residual scan, and decision
# ============================================================

primary_ids = set(PRIMARY_CANDIDATES)
primary_structures = structure_manifest[
    structure_manifest["candidate_id"].isin(primary_ids)
]
primary_vasp = vasp_manifest[
    vasp_manifest["candidate_id"].isin(primary_ids)
]
primary_cases = case_manifest[
    case_manifest["candidate_id"].isin(primary_ids)
]

complete_primary_pairs = int(
    primary_structures.groupby("candidate_id")["written"].all().sum()
)
ordered_primary_pairs = int(
    primary_structures.groupby("candidate_id")["is_ordered"].all().sum()
)
manageable_primary_pairs = int(
    primary_cases["manageable_cell"].fillna(False).sum()
)
vasp_primary_pairs = int(
    primary_vasp.groupby("candidate_id")["template_written"].all().sum()
)

licensed_potcars = list(OUTPUT_ROOT.rglob("POTCAR"))
for path in list(licensed_potcars):
    if path.name == "POTCAR.spec":
        licensed_potcars.remove(path)

# Private output scan. This scan detects accidental secrets and unrelated legacy framing.
scan_patterns = {
    "api_key_assignment": re.compile(
        r"(?i)(api[_-]?key|token|secret)\s*[:=]\s*['\"][A-Za-z0-9_-]{16,}"
    ),
    "previous_repository_identifier": re.compile(
        r"(?i)legacy[_ -]?repository[_ -]?identifier"
    ),
}
scan_rows = []
for path in OUTPUT_ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in {
        ".csv", ".json", ".txt", ".md", ".yaml", ".yml", ".py", ".ipynb",
        ".incar", ".spec",
    }:
        continue
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    for pattern_name, pattern in scan_patterns.items():
        matches = pattern.findall(text)
        scan_rows.append({
            "relative_path": str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/"),
            "pattern": pattern_name,
            "n_matches": len(matches),
            "pass": len(matches) == 0,
        })

residual_scan = pd.DataFrame(scan_rows)
if residual_scan.empty:
    residual_scan = pd.DataFrame([
        {
            "relative_path": "",
            "pattern": "no_text_files_scanned",
            "n_matches": 0,
            "pass": True,
        }
    ])
residual_scan.to_csv(
    AUDIT_DIR / "11_private_output_residual_and_secret_scan.csv",
    index=False,
)

gate_rows = [
    ("notebook08_and_06_decisions_accepted", bool(accepted08 and accepted10c)),
    ("all_locked_core_mappings_exact", bool(core_mapping_audit["pass"].all())),
    ("all_raw_records_resolved_by_both_exact_ids", bool(raw_record_audit["pass"].all())),
    ("at_least_three_complete_primary_pairs", complete_primary_pairs >= 3),
    ("all_five_complete_primary_pairs", complete_primary_pairs == 5),
    ("at_least_three_ordered_primary_pairs", ordered_primary_pairs >= 3),
    ("at_least_three_manageable_primary_pairs", manageable_primary_pairs >= 3),
    ("at_least_three_vasp_primary_pairs_written", vasp_primary_pairs >= 3),
    ("no_licensed_potcar_files_written", len(licensed_potcars) == 0),
    ("secret_and_residual_scan_pass", bool(residual_scan["pass"].all())),
]
gates = pd.DataFrame(gate_rows, columns=["gate", "pass"])
gates.to_csv(
    AUDIT_DIR / "11_go_no_go_gate_audit.csv",
    index=False,
)

if (
    complete_primary_pairs == 5
    and ordered_primary_pairs == 5
    and manageable_primary_pairs == 5
    and vasp_primary_pairs == 5
    and gates["pass"].all()
):
    final_decision = "FULL_GO_TO_PILOT_DFT"
elif (
    complete_primary_pairs >= 3
    and ordered_primary_pairs >= 3
    and manageable_primary_pairs >= 3
    and vasp_primary_pairs >= 3
    and bool(gates.loc[
        ~gates["gate"].eq("all_five_complete_primary_pairs"),
        "pass",
    ].all())
):
    final_decision = "GO_WITH_RESTRICTIONS_TO_PILOT_DFT"
else:
    final_decision = "HOLD_EXACT_STRUCTURE_TRACEABILITY_REQUIRED"

decision_payload = {
    "final_decision": final_decision,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "primary_candidate_count": len(PRIMARY_CANDIDATES),
    "complete_primary_pairs": complete_primary_pairs,
    "ordered_primary_pairs": ordered_primary_pairs,
    "manageable_primary_pairs": manageable_primary_pairs,
    "vasp_primary_pairs_written": vasp_primary_pairs,
    "optional_control": OPTIONAL_CONTROL,
    "exact_structure_source": "Notebook 01 raw Na insertion-electrode records",
    "formula_fallback_used": False,
    "materials_project_api_used": False,
    "dft_calculations_run": False,
    "next_action": (
        "Run one charged/discharged pilot pair plus a compatible metallic-Na reference."
        if final_decision in {
            "FULL_GO_TO_PILOT_DFT",
            "GO_WITH_RESTRICTIONS_TO_PILOT_DFT",
        }
        else "Resolve failed exact-structure or input-generation gates before DFT."
    ),
}
(METADATA_DIR / "11_final_decision.json").write_text(
    json.dumps(decision_payload, indent=2),
    encoding="utf-8",
)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "pymatgen": getattr(pymatgen, "__version__", "unknown"),
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
}
(METADATA_DIR / "11_software_environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

# Manifest excludes itself.
manifest_rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and path.name != "11_output_file_manifest.csv":
        manifest_rows.append({
            "relative_path": str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/"),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        })
output_manifest = pd.DataFrame(manifest_rows)
output_manifest.to_csv(
    METADATA_DIR / "11_output_file_manifest.csv",
    index=False,
)

display(gates)
print("FINAL DECISION:", final_decision)

if final_decision == "HOLD_EXACT_STRUCTURE_TRACEABILITY_REQUIRED":
    raise RuntimeError(final_decision)